# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kenzo4k/Flyrank-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### 7-Feature Contract Frame (Lane 3: Structured Content Archetype Clustering)
Our unsupervised clustering model uses exactly **7 continuous numerical features**, each derived from observable search, freshness, depth, and engagement metrics. No categorical features or text embeddings are used.

| # | Model Feature | Raw Source Column | Transformation |
|---|---|---|---|
| 1 | `log_impressions` | `impressions_90d` | $\ln(1 + x)$ — compresses the heavy right tail (median 731, mean ~5,200). |
| 2 | `avg_position` | `avg_position` | $0 \rightarrow 100$ imputation — 1,205 rows have `avg_position == 0` meaning "no GSC data", not rank zero. Imputed to 100.0 (deep off-SERP). |
| 3 | `ctr` | `ctr` | No transformation — already a ×100 percentage (0.76 means 0.76%). |
| 4 | `days_since_last_update` | `days_since_last_update` | No transformation — integer days. |
| 5 | `content_age_days` | `content_age_days` | No transformation — integer days since first publish. |
| 6 | `word_count` | `word_count` | Median-imputed — 25.7% missing (follows `content_type`). |
| 7 | `engagement_rate` | `engagement_rate` | Zero-filled — 72.1% of rows are zero due to GA4 tracking gaps across clients. |

In [ ]:
# Section 1 Code: Setup, Data Ingestion, and Feature Vector Construction
import os
import sys
import subprocess
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kenzo4k/Flyrank-ML"
REPO_DIR = "Flyrank-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

data_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in data_candidates if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

print(f"Loaded dataset: {len(df):,} rows x {len(df.columns)} columns across {df['client_id'].nunique()} clients.")

# Build the 7-feature contract frame
clustering_features = [
    'log_impressions', 'avg_position', 'ctr',
    'days_since_last_update', 'content_age_days',
    'word_count', 'engagement_rate'
]

X = pd.DataFrame(index=df.index)
X['log_impressions'] = np.log1p(df['impressions_90d'])
X['avg_position'] = df['avg_position'].replace(0, 100.0)
X['ctr'] = df['ctr']
X['days_since_last_update'] = df['days_since_last_update']
X['content_age_days'] = df['content_age_days']
X['word_count'] = df['word_count'].fillna(df['word_count'].median())
X['engagement_rate'] = df['engagement_rate'].fillna(0.0)

print(f"\nFeature matrix shape: {X[clustering_features].shape}")
print(f"Features: {clustering_features}")
print(f"\nFeature dtypes:\n{X[clustering_features].dtypes}")

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature-by-Feature Notes

| Feature | Meaning | Missing Handling | Available Before Prediction? |
|---|---|---|---|
| `log_impressions` | Log-transformed 90-day organic search impressions. Measures search demand/visibility. | No missing values (source `impressions_90d` is complete). | ✅ Yes — trailing 90-day aggregate, strictly historical. |
| `avg_position` | Mean Google Search Console ranking position over 90 days. Lower = higher rank. | 1,205 rows have raw value 0 ("no data"), imputed to 100.0 (deep off-SERP). | ✅ Yes — trailing 90-day aggregate. |
| `ctr` | Click-through rate from search results (×100 percentage). | No missing values. | ✅ Yes — trailing 90-day aggregate. |
| `days_since_last_update` | Integer days since the content was last editorially updated. | No missing values. | ✅ Yes — a calendar fact knowable at any point. |
| `content_age_days` | Integer days since original publication date. | No missing values. | ✅ Yes — a calendar fact knowable at any point. |
| `word_count` | Article body word count. | 25.7% missing (follows `content_type`). Median-imputed. | ✅ Yes — a static content attribute. |
| `engagement_rate` | GA4 engaged sessions / total sessions × 100. | ~72% zero due to GA4 tracking gaps. Zero-filled (not median) to avoid injecting a false signal. | ✅ Yes — trailing aggregate from GA4. |

> **Temporal Safety**: All 7 features are trailing 90-day aggregates or static content attributes. None require knowledge of the label window (last 30 vs preceding 30 days). The timeline is clean: **features ← observation window → label**.

In [ ]:
# Section 2 Code: Feature Notes Verification — Missing Values, Dtypes, Descriptive Statistics
print("=" * 75)
print("FEATURE NOTES: MISSING VALUES AND DESCRIPTIVE STATISTICS")
print("=" * 75)

# Missing values in raw source columns
raw_source_cols = ['impressions_90d', 'avg_position', 'ctr',
                   'days_since_last_update', 'content_age_days',
                   'word_count', 'engagement_rate']
print("\nRaw Source Column Missing Counts:")
for col in raw_source_cols:
    n_miss = df[col].isna().sum()
    pct_miss = n_miss / len(df) * 100
    print(f"  {col:30s} : {n_miss:,} missing ({pct_miss:.1f}%)")

# Verify zero missing in the constructed feature matrix
print(f"\nConstructed Feature Matrix Missing Counts (post-imputation):")
for col in clustering_features:
    n_miss = X[col].isna().sum()
    print(f"  {col:30s} : {n_miss}")

assert X[clustering_features].isna().sum().sum() == 0, "Missing values remain in feature matrix!"
print("\nMissing Value Check: PASSED (Zero NaN in feature matrix after imputation).")

# Descriptive statistics
print("\n" + "=" * 75)
print("DESCRIPTIVE STATISTICS (Constructed Feature Matrix)")
print("=" * 75)
print(X[clustering_features].describe().round(2).to_string())

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Three-Category Leakage Taxonomy
Per `skills/hunting-leakage-and-validating/SKILL.md`, we systematically attack our own features across all three leakage pathways:

1. **Label-Derived Features**: The label `is_declining_label` is computed FROM `trend_direction`, which is computed FROM `trend_pct`. Therefore `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d`, and `april_imp` are strictly prohibited — any of these in the feature set would let the model read the answer during training.

2. **Future / Overlapping Windows**: The 7 features are all trailing 90-day aggregates or static attributes. The label compares the *last* 30 days vs. the *preceding* 30 days within that 90-day window. No feature directly encodes the label's comparison window, so this pathway is clear.

3. **Decision-Derived Product Flags**: Columns like `health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, and tier columns (`impression_tier`, `position_tier`, `age_tier`, `freshness_tier`) encode decisions from FlyRank's existing rule engine. Using them as features would mean learning the old system's rules, not the actual data patterns. They may serve as a baseline to beat, never as model inputs.

### Controlled Leakage Trap Experiment
To prove our audit harness is sensitive (per the skill: *"deliberately ADD a leaky feature and watch the score jump toward 1.0"*), we train a classifier to predict `is_declining_label` under a grouped client split:
- **Clean Model** (7 contract features): measures honest out-of-domain AUC.
- **Poisoned Model** (7 contract features + leaky `trend_pct`): should instantly confess with near-perfect AUC.

In [ ]:
# Section 3 Code: Automated Leakage Hunt and Controlled Trap Experiment
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# --- Step 1: Automated Feature Overlap Assertion ---
forbidden_outcomes = ['trend_direction', 'trend_pct', 'is_declining_label',
                      'impressions_last_30d', 'impressions_prev_30d', 'april_imp']
forbidden_product_flags = ['health_score', 'priority_score', 'action_type',
                           'needs_ctr_fix', 'impression_tier', 'position_tier',
                           'age_tier', 'freshness_tier']
forbidden_identifiers = ['client_id', 'content_id']

all_forbidden = forbidden_outcomes + forbidden_product_flags + forbidden_identifiers

overlap = set(clustering_features).intersection(set(all_forbidden))

print("=" * 75)
print("AUTOMATED FEATURE LEAKAGE HUNT")
print("=" * 75)
print(f"Features in Model Pipeline  ({len(clustering_features)}) : {clustering_features}")
print(f"Forbidden Columns Checked   ({len(all_forbidden)}) : {all_forbidden}")
print(f"Detected Feature Overlap    : {list(overlap)}")

assert len(overlap) == 0, f"CRITICAL LEAKAGE: Prohibited columns found in model features: {overlap}"
print("\nLeakage Hunt Status: PASSED (Zero prohibited columns in feature matrix).")

# --- Step 2: Grouped Client Split for Trap Experiment ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=df['client_id']))

# --- Step 3: Clean Model (7 Contract Features) ---
scaler_clean = StandardScaler()
X_clean_train = scaler_clean.fit_transform(X.iloc[train_idx][clustering_features])
X_clean_test = scaler_clean.transform(X.iloc[test_idx][clustering_features])

clf_clean = LogisticRegression(random_state=42, max_iter=1000)
clf_clean.fit(X_clean_train, df.iloc[train_idx]['is_declining_label'])
y_pred_clean = clf_clean.predict_proba(X_clean_test)[:, 1]
auc_clean = roc_auc_score(df.iloc[test_idx]['is_declining_label'], y_pred_clean)

# --- Step 4: Poisoned Model (7 Features + Leaky trend_pct) ---
X_poisoned_train = X.iloc[train_idx][clustering_features].copy()
X_poisoned_train['trend_pct'] = df.iloc[train_idx]['trend_pct'].fillna(0.0)
X_poisoned_test = X.iloc[test_idx][clustering_features].copy()
X_poisoned_test['trend_pct'] = df.iloc[test_idx]['trend_pct'].fillna(0.0)

scaler_poison = StandardScaler()
X_p_tr_sc = scaler_poison.fit_transform(X_poisoned_train)
X_p_te_sc = scaler_poison.transform(X_poisoned_test)

clf_poison = LogisticRegression(random_state=42, max_iter=1000)
clf_poison.fit(X_p_tr_sc, df.iloc[train_idx]['is_declining_label'])
y_pred_poison = clf_poison.predict_proba(X_p_te_sc)[:, 1]
auc_poison = roc_auc_score(df.iloc[test_idx]['is_declining_label'], y_pred_poison)

print("\n" + "=" * 75)
print("CONTROLLED LEAKAGE TRAP EXPERIMENT")
print("=" * 75)
print(f" - Clean Feature Set ROC-AUC   : {auc_clean:.4f} (Honest Baseline)")
print(f" - Poisoned (+ trend_pct) AUC  : {auc_poison:.4f} (Instant Confession: Harness Catches Leakage)")
print(f" - AUC Jump                    : +{auc_poison - auc_clean:.4f}")
print("\nVerdict: The audit harness is sensitive — injecting a leaky feature causes an")
print("immediate jump toward 1.0, confirming that our clean feature set is honest.")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Column Inventory

**Category 1 — Future / Outcome Metrics (Target Leakage)**:
- `trend_direction` — Directly encodes the label (`is_declining_label` is derived from this column).
- `trend_pct` — The numerical value from which `trend_direction` is computed; using it is reading the answer.
- `is_declining_label` — The evaluative benchmark itself; strictly reserved for downstream validation.
- `impressions_last_30d` — Part of the label's comparison window (last 30 vs preceding 30 days).
- `impressions_prev_30d` — Part of the label's comparison window.
- `april_imp` — A single-month snapshot that overlaps the label computation period.

**Category 2 — Decision-Derived Product Flags (Circular Logic)**:
- `health_score` — A composite heuristic score from FlyRank's existing system; learning it = learning the old rule.
- `priority_score` — Another existing-system heuristic composite.
- `action_type` — An action recommendation from the existing system (the decision we're trying to improve).
- `needs_ctr_fix` — A binary flag from the existing CTR-fix heuristic rule.
- `impression_tier`, `position_tier`, `age_tier`, `freshness_tier` — Hand-crafted categorical bins from the existing system; they encode prior business decisions, not raw observations.

**Category 3 — Identifiers (Memorization Vectors)**:
- `client_id` — Pseudonymized client identifier. Used only for `GroupShuffleSplit` cross-validation, never as a feature.
- `content_id` — Pseudonymized content identifier. Used only for joins and queue output.

In [ ]:
# Section 4 Code: Final Exclusion List Verification
print("=" * 75)
print("EXCLUDED COLUMN INVENTORY — FINAL VERIFICATION")
print("=" * 75)

exclusion_inventory = {
    'Future / Outcome Metrics': forbidden_outcomes,
    'Decision-Derived Product Flags': forbidden_product_flags,
    'Identifiers (Memorization Vectors)': forbidden_identifiers
}

for category, cols in exclusion_inventory.items():
    print(f"\n{category}:")
    for col in cols:
        in_data = col in df.columns
        in_features = col in clustering_features
        status = "EXISTS in dataset" if in_data else "NOT in dataset"
        leak_status = "IN FEATURES (LEAK!)" if in_features else "excluded"
        print(f"  {col:30s} — {status:25s} — {leak_status}")

# Final assertion
final_overlap = set(clustering_features).intersection(set(all_forbidden))
assert len(final_overlap) == 0, f"LEAKAGE DETECTED: {final_overlap}"
print(f"\nFinal Leakage Check: PASSED")
print(f"Model uses {len(clustering_features)} features, excludes {len(all_forbidden)} forbidden columns.")
print(f"Zero overlap confirmed.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.